# IV. Standardisation des variables #

## 1. MOTIFS D'ADMISSION

cf notebook " pec motif"

In [ ]:
import pandas as pd
import numpy as np

In [ ]:


# Chargement du fichier
df_imputed = pd.read_csv("df_final_binaire_imputed.csv", sep=',')

## 2. Others variables ##

In [ ]:
# --- GROUPEMENT DES LABOS (Logique "AU MOINS UN") --- a mettre dans autre note book cleaning harmonizing
groupes_labo = {
    'is_bilan_routine': ['is_hb', 'is_leuco', 'is_formule_leuco', 'is_urea', 'is_creat',
                      'is_sodium', 'is_potassium', 'calcium' 'is_plqt', 'is_tp', 'is_tca',
                      'is_calcium', 'is_ck'],
    'is_bilan_cardio_vasc': ['is_tropo', 'is_bnp', 'is_ckmb', 'is_ddimere'],
    'is_bilan_infectieux': ['is_crp', 'is_pct', 'is_culture'],
    'is_bilan_hepato_dig': ['is_alat', 'is_asat', 'is_bili_tot', 'is_lipase', 'is_alp'],
    'is_bilan_martial': ['is_fer', 'is_ferritine'],
    'is_bilan_coag_specifique' : ['is_fibrinogen', 'is_aXa/aIIa']
}

for nom_groupe, cols in groupes_labo.items():
    # Correction ici : on vérifie dans df_imputed directement
    cols_presentes = [c for c in cols if c in df_imputed.columns]

    if cols_presentes:
        # On crée le groupe : 1 si au moins une colonne du groupe est à 1
        df_imputed[nom_groupe] = df_imputed[cols_presentes].max(axis=1).astype(int)
        print(f"✅ Groupe créé : {nom_groupe} (basé sur {len(cols_presentes)} colonnes)")
    else:
        print(f"⚠️ Aucune colonne trouvée pour le groupe : {nom_groupe}")

# --- MISE À JOUR DES LISTES DE COLONNES ---
toutes_cols_detail = [item for sublist in groupes_labo.values() for item in sublist]
# On ne garde que celles qui existent vraiment pour éviter les erreurs plus tard
toutes_cols_detail = [c for c in toutes_cols_detail if c in df_imputed.columns]

In [ ]:
#===============================================================
# STANDARDISATION
#===============================================================

from sklearn.preprocessing import PowerTransformer, StandardScaler
import pandas as pd

# 1. Configuration des groupes (Tes listes complètes)
quanti_cols = [
    'age', 'tas', 'tad', 'fc', 'temp', 'sat', 'fr', 'o2_flow',
    'gcs', 'hgt_mmol_L', 'pupil_right', 'pupil_left', 'ethylotest',
    'hemocue', 'eval_douleur', 'duree_triage_ioa_min'
] # voior quoi faire avec ces pupilles

binary_cols = [
    #'sexe_binr',
    'is_ta_measured', 'is_fc_measured', 'is_temp_measured',
    'is_sat_measured', 'is_fr_measured', 'has_o2_support', 'is_gcs_measured',
    'is_hgt_mmol_L_measured', 'is_pupil_right_measured', 'is_urine_dipstick_clean_measured',
    'is_eval_douleur_measured', 'is_ethylotest_measured', 'is_hemocue_measured',
    'has_echo', 'has_scanner', 'has_radio_conv', 'has_irm',

    'is_bilan_routine',  'is_bilan_cardio_vasc', 'is_bilan_infectieux', 'is_bilan_hepato_dig', 'is_calcium_ionise', 'is_bilan_martial', 'is_bilan_coag_specifique',

    'is_gds', 'is_lcr', 'is_lactates'
]

quali_cols = [
    #'motif_topic',
    'transport_grouped', 'devenir']

# 2. Création du DataFrame de travail
# On s'assure de ne prendre que ce dont on a besoin
df_subset = df_imputed[quanti_cols + binary_cols + quali_cols].copy()

# 3.
# ÉTAPE A : Yeo-Johnson sur quanti SEULEMENT (avec standardize=True par défaut)
pt = PowerTransformer(method='yeo-johnson', standardize=True)
df_subset[quanti_cols] = pt.fit_transform(df_subset[quanti_cols])

# ÉTAPE B : Dummies sur quali
df_final = pd.get_dummies(df_subset, columns=quali_cols, drop_first=True)
df_final = df_final.astype(float)

# ÉTAPE C : StandardScaler UNIQUEMENT sur les binaires
# (les quanti sont déjà standardisées par Yeo-Johnson)
#scaler = StandardScaler()
#df_final[binary_cols] = scaler.fit_transform(df_final[binary_cols])
# NE PAS SCALER LES BINAIRES PARCE QUE SINON CA INTRODUIT UN POIDS TROP IMPORTANT SUR CES VARIABLES PAR RAPPORT AUX QUANTI (ET CA FAIT PLANTER L'UMAP)




df_scaled = df_final.copy()
X_scaled = df_scaled.values   # ← la matrice numpy comme avant


# --- BILAN ---
print(f"✅ Matrice finale prête.")
print(f"Total de lignes (patients) : {X_scaled.shape[0]}")
print(f"Total de colonnes (features) : {X_scaled.shape[1]}")
print("-" * 30)
# Correction des noms pour tes prints de vérification
print(f"Variables Quantitatives (Yeo-Johnson) : {len(quanti_cols)}")
print(f"Variables Binaires : {len(binary_cols)}")

In [ ]:
# --- VÉRIFICATION DE LA PERTINENCE DES BINAIRES ---
# 1. On récupère toutes les colonnes qui sont binaires dans df_final
# (Celles qui n'ont que 0.0 et 1.0 comme valeurs)
all_bin_cols = [c for c in df_final.columns if df_final[c].nunique() <= 2]

# 2. Calcul du taux de présence (moyenne * 100)
bin_stats = (df_final[all_bin_cols].mean() * 100).sort_values(ascending=False)

# 3. Affichage des résultats
print("🔍 ANALYSE DE PERTINENCE DES BINAIRES (Dummies inclus)")
print("="*60)
print(f"{'Variable':<45} | {'Présence (%)':<15}")
print("-" * 65)

for col, val in bin_stats.items():
    # On met une alerte si la variable est trop rare (< 0.5%) ou trop commune (> 99.5%)
    status = ""
    if val < 0.5: status = "⚠️ TROP RARE"
    if val > 99.5: status = "⚠️ TROP COMMUNE"

    print(f"{col:<45} | {val:>12.2f}%  {status}")

# 4. Identification automatique des variables "inutiles" (variance quasi nulle)
to_drop = bin_stats[(bin_stats < 0.1) | (bin_stats > 99.9)].index.tolist()
if to_drop:
    print("\n🚨 VARIABLES À POTENTIELLEMENT SUPPRIMER (Variance nulle) :")
    print(to_drop)

In [ ]:
# ================================================================
# CHECK-UP COMPLET DU DATAFRAME SCALÉ
# ================================================================

def check_df_readiness(df):
    print(f"📊 Dimensions : {df.shape[0]} patients, {df.shape[1]} variables")
    print("-" * 50)

    # 1. Vérification des types de données
    dtypes_count = df.dtypes.value_counts()
    print(f"🧐 Types de colonnes :\n{dtypes_count}")

    # 2. Identification des colonnes non-numériques (devrait être vide !)
    non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        print(f"🚨 ALERTE : Colonnes non-numériques trouvées : {non_numeric}")
    else:
        print("✅ Toutes les colonnes sont bien numériques.")

    # 3. Vérification des valeurs manquantes (devrait être 0 après imputation)
    missing = df.isnull().sum().sum()
    if missing > 0:
        print(f"🚨 ALERTE : Il reste {missing} valeurs manquantes !")
    else:
        print("✅ Aucune valeur manquante.")

    # 4. Analyse de la distribution des types (Num vs Bin)
    # On considère binaire si seulement 2 valeurs (0 et 1 généralement)
    bin_cols = [c for c in df.columns if df[c].nunique() <= 2]
    num_cols = [c for c in df.columns if c not in bin_cols]

    print("-" * 50)
    print(f"🔢 Variables continues détectées : {len(num_cols)}")
    print(f"🔘 Variables binaires détectées : {len(bin_cols)}")

    # 5. Détection des variables "mortes" (variance nulle)
    constant_cols = [c for c in df.columns if df[c].nunique() <= 1]
    if constant_cols:
        print(f"💀 Colonnes constantes (à supprimer !) : {constant_cols}")
    else:
        print("✅ Aucune colonne constante.")

# Lancer la vérification
check_df_readiness(df_scaled)

In [ ]:
# Vérifie si les colonnes de labo "Détail" sont toujours là ou si elles ont été exclues
detail_cols_check = [c for c in toutes_cols_detail if c in df_scaled.columns]
print(f"Colonnes de détail encore présentes : {len(detail_cols_check)}")

# Si ce chiffre est > 0, c'est que tu as des doublons (le groupe ET le détail).
# Pour ton clustering, il vaut mieux les enlever et ne garder que les groupes.

In [ ]:
#!pip install "numpy<=2.3" # umap ne fonctionne pas avec numpy 2.4, il faut 2.3 ou inferieur


import numpy as np
import umap
import matplotlib.pyplot as plt

In [ ]:
# investigation si il reste des valeurs manquantes
import numpy as np
import pandas as pd

# --- A. Vérification dans le DataFrame final ---
print("--- Analyse des NaNs par colonne dans df_final ---")
nan_summary = df_final.isna().sum()
nan_only = nan_summary[nan_summary > 0]

if nan_only.empty:
    print("✅ Aucune colonne de df_final ne contient de NaN.")
else:
    print("❌ Colonnes problématiques trouvées :")
    print(nan_only)

# --- B. Vérification dans la matrice X_scaled (celle qui fait planter l'UMAP) ---
# On vérifie s'il y a des NaNs ou des valeurs infinies
if np.any(np.isnan(X_scaled)):
    print(f"\n❌ ERREUR : Il y a {np.isnan(X_scaled).sum()} valeurs NaN dans X_scaled !")

if np.any(np.isinf(X_scaled)):
    print(f"❌ ERREUR : Il y a {np.isinf(X_scaled).sum()} valeurs INFINIES dans X_scaled !")

if not np.any(np.isnan(X_scaled)) and not np.any(np.isinf(X_scaled)):
    print("\n✅ X_scaled est mathématiquement parfait (pas de NaN ni d'Inf).")

In [ ]:
# 1. On rassemble toutes les colonnes qui vont entrer dans l'UMAP
all_clustering_cols = quanti_cols + binary_cols + quali_cols

# 2. On filtre pour ne garder que celles qui existent vraiment dans le DF (sécurité)
existing_cols = [c for c in all_clustering_cols if c in df_imputed.columns]

# 3. On crée un DataFrame qui ne contient que les lignes avec au moins un NaN
# mais on ne regarde QUE dans nos variables cibles
df_nan_debug = df_imputed[df_imputed[existing_cols].isna().any(axis=1)]

print(f"🔍 Nombre de patients ayant au moins un NaN dans les variables de clustering : {len(df_nan_debug)}")

if len(df_nan_debug) > 0:
    # On identifie les colonnes précises qui posent problème
    cols_with_nans = [c for c in existing_cols if df_imputed[c].isna().any()]
    print(f"❌ Colonnes contenant des NaNs : {cols_with_nans}")

    # On affiche les premières lignes pour comprendre (juste les colonnes à NaNs)
    display(df_nan_debug[cols_with_nans].head(10))
else:
    print("✅ Parfait ! Aucune de tes variables de clustering ne contient de NaN.")

In [ ]:
import os
print(os.cpu_count())  # nombre de cœurs dispo

In [ ]:
# # =====================================================================
# # AVEC TOUTES  LES VARIABLES (sauf motif a rajouter)
# #================================================================
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import umap.umap_ as umap  # Attention à l'import spécifique pour UMAP
# import seaborn as sns  # Optionnel, mais aide souvent pour le style
#
# # 1. Création du réducteur UMAP 2D
# # n_neighbors : balance entre structure locale et globale
# # min_dist : contrôle le tassement des points (0.1 est standard)
# reducer_2d = umap.UMAP(
#     n_neighbors=15,
#     min_dist=0.1,
#     n_components=2,
#     random_state=42,
#     init='random',        # ← évite le warning spectral
#     n_jobs = os.cpu_count() - 1, # 255,            # ← utilise tous les cœurs dispo sauf 1  (vos 256 - 1 !)
#     low_memory=False,     # ← plus rapide si vous avez assez de RAM
# )
#
# # 2. Calcul des coordonnées
# embedding_2d = reducer_2d.fit_transform(X_scaled)
#
# # 3. Affichage
# plt.figure(figsize=(12, 8))
# plt.scatter(
#     embedding_2d[:, 0],
#     embedding_2d[:, 1],
#     s=15,  # Taille des points
#     alpha=0.6,  # Transparence pour voir les densités
#     c='indigo',  # Couleur unique pour l'instant
#     edgecolors='none'
# )
#
# plt.title("Carte UMAP des Urgences (Structure Naturelle des Patients)", fontsize=14)
# plt.xlabel("Dimension UMAP 1")
# plt.ylabel("Dimension UMAP 2")
# plt.grid(True, linestyle='--', alpha=0.3)
#


In [ ]:
# # ======================================================
# # GMM BIC — version rapide
# # =========================================================
# from sklearn.mixture import GaussianMixture
# import matplotlib.pyplot as plt
#
# n_components = range(1, 15)
# bics = []
#
# for n in n_components:
#     gmm = GaussianMixture(
#         n_components=n,
#         covariance_type='diag',  # ← rapide
#         max_iter=100,
#         n_init=1,
#         random_state=42
#     )
#     gmm.fit(embedding_2d)
#     bics.append(gmm.bic(embedding_2d))
#     print(f"n={n} ✓")
#
# plt.plot(n_components, bics, 'bo-')
# plt.xlabel('Nombre de clusters')
# plt.ylabel('BIC Score')
# plt.title('Recherche du nombre optimal de clusters (BIC)')
# plt.show()

In [ ]:
# from sklearn.mixture import BayesianGaussianMixture
#
# # 1. Clustering sur l'embedding 2D (qui vient de tes variables triage)
# bgmm = BayesianGaussianMixture(
#     n_components=10,
#     weight_concentration_prior=1e-3,
#     random_state=42
# )
#
# # On calcule les labels
# labels = bgmm.fit_predict(embedding_2d)
#
# # 2. On ajoute les labels au DataFrame (C'est là qu'on fait l'audit)
# # On s'assure d'utiliser df_scaled pour avoir les noms de colonnes
# df_scaled['cluster_gmm'] = labels
#
# # 3. Visualisation (on peut utiliser embedding_2d directement)
# plt.figure(figsize=(10, 7))
# plt.scatter(
#     embedding_2d[:, 0],
#     embedding_2d[:, 1],
#     c=labels,
#     cmap='tab10',
#     s=10,
#     alpha=0.6
# )
# plt.colorbar(label='Cluster ID')
# plt.title("Clustering GMM sur Variables Triage")
# plt.show()
#
# # 4. L'AUDIT : La partie cruciale
# # On calcule la moyenne de CHAQUE variable originale par cluster
# audit = df_scaled.groupby('cluster_gmm').mean()
#
# print("📊 PROFILAGE DES CLUSTERS :")
# # On transpose (T) pour avoir les variables en lignes et on colore
# # On utilise slice(None) pour ne pas essayer de colorer la colonne 'cluster_gmm' elle-même
# display(audit.T.style.background_gradient(axis=1, cmap='YlOrRd'))

In [ ]:
# # DBSCAN
#
# from sklearn.cluster import DBSCAN
#
# # On essaye d'abord un eps qui capture les filaments
# # Si ton graph K-NN (le précédent) montrait un coude vers 0.3 ou 0.4 :
# db = DBSCAN(eps=0.4, min_samples=15)
# clusters = db.fit_predict(embedding_2d)
#
# # Visualisation
# plt.figure(figsize=(12, 8), facecolor='black')  # Fond de figure propre
# plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=clusters, cmap='Spectral', s=5)
# plt.grid(True, linestyle='-', color='#f0f0f0', alpha=0.5, linewidth=0.5)
# plt.gca().set_axisbelow(True)  # Pour que la grille passe DERRIÈRE les points
# plt.title(f"Clusters trouvés par DBSCAN : {len(np.unique(clusters)) - 1}")
# plt.show()
#
# # 1. Lancer DBSCAN sur ton embedding
# # Ajuste eps selon ton graphique de coude (ex: 0.3 ou 0.5)
# db = DBSCAN(eps=0.4, min_samples=15)
# df_scaled['cluster_dbscan'] = clusters
#
# # 2. Nettoyage des doublons (pour éviter le KeyError précédent)
# df_audit_db = df_scaled.loc[:, ~df_scaled.columns.duplicated()].copy()
#
# # 3. Calcul de l'audit
# audit_db = df_audit_db.groupby('cluster_dbscan').mean()
#
# # 4. Affichage
# print(f"📊 Audit DBSCAN : {len(audit_db)} groupes détectés (le -1 est le bruit)")
# display(audit_db.T.style.background_gradient(axis=1, cmap='coolwarm'))
#
# from scipy.cluster.hierarchy import dendrogram, linkage
#
# # On utilise la méthode 'ward' ou 'complete' sur tes coordonnées UMAP
# Z = linkage(embedding_2d, method='ward')
#
# plt.figure(figsize=(12, 6), facecolor='black')
# dendrogram(Z, truncate_mode='lastp', p=20)  # On affiche les 20 derniers regroupements
# # --- GRILLE HORIZONTALE DISCRÈTE SEULEMENT ---
# plt.gca().yaxis.grid(True, linestyle='--', color='#e0e0e0', alpha=0.4)
# plt.gca().xaxis.grid(False)  # On enlève la grille verticale souvent inutile ici
#
# plt.title("Dendrogramme : Aide au choix du nombre de clusters")
# plt.ylabel("Distance de fusion")
# plt.axhline(y=15, color='r', linestyle='--')  # Exemple de ligne de coupe
# plt.show()
#


In [ ]:
# # ===========================================================
# # AVEC VARIABLE TRIAGE SEULEMENT (SAUF TRI EVIDEMMENT) BINAIRE
# # ================================================================
#
# base_cols = [
#     'age', 'sexe_binr', 'is_ta_measured', 'is_fc_measured', 'is_temp_measured',
#     'is_sat_measured', 'is_fr_measured', 'has_o2_support', 'is_gcs_measured',
#     'is_hgt_mmol_L_measured', 'is_pupil_right_measured',
#     'is_urine_dipstick_clean_measured', 'is_eval_douleur_measured',
#     'is_ethylotest_measured', 'is_hemocue_measured'
# ]
#
# # Ajouter les dummies de transport
# transport_cols = [c for c in df_scaled.columns if
#                   c.startswith("transport_grouped_")]  # VOIR POUR MOTIF A RAJOUTER UNE FOIS TROUVER UNE SOLUTION
#
# umap_cols = base_cols + transport_cols
#
# df_umap = df_scaled[umap_cols]
#
# # 1. Création du réducteur UMAP 2D
# # n_neighbors : balance entre structure locale et globale
# # min_dist : contrôle le tassement des points (0.1 est standard)
# reducer_2d = umap.UMAP(
#     n_neighbors=15,
#     min_dist=0.1,
#     n_components=2,
#     random_state=42,
#     init = 'random'  # ← pour éviter les artefacts de l'init spectral sur des données binaires
# )
#
# # 2. Calcul des coordonnées
# embedding_2d = reducer_2d.fit_transform(df_umap)
#
# # 3. Affichage
# plt.figure(figsize=(12, 8))
# plt.scatter(
#     embedding_2d[:, 0],
#     embedding_2d[:, 1],
#     s=15,  # Taille des points
#     alpha=0.6,  # Transparence pour voir les densités
#     c='indigo',  # Couleur unique pour l'instant
#     edgecolors='none'
# )
#
# plt.title("Carte UMAP des Urgences (Structure Naturelle des Patients)", fontsize=14)
# plt.xlabel("Dimension UMAP 1")
# plt.ylabel("Dimension UMAP 2")
# plt.grid(True, linestyle='--', alpha=0.3)
# plt.show()

In [ ]:
# from sklearn.mixture import GaussianMixture
# import matplotlib.pyplot as plt
#
# n_components = range(1, 15)
# bics = []
#
# for n in n_components:
#     gmm = GaussianMixture(n_components=n, random_state=42)
#     gmm.fit(embedding_2d)  # On teste sur tes coordonnées UMAP
#     bics.append(gmm.bic(embedding_2d))
#
# plt.plot(n_components, bics, 'bo-')
# plt.xlabel('Nombre de clusters')
# plt.ylabel('BIC Score')  # Le point le plus bas est le meilleur
# plt.title('Recherche du nombre optimal de clusters (BIC)')
# plt.show()

In [ ]:
# # GMM
#
#
# from sklearn.mixture import BayesianGaussianMixture
#
# # 1. Ajustement du modèle sur tes coordonnées UMAP
# # On en demande 10, le modèle "Bayesian" filtrera ceux qui ne sont pas assez denses
# bgmm = BayesianGaussianMixture(
#     n_components=10,
#     weight_concentration_prior=1e-3,
#     random_state=42
# )
# df_umap['cluster_gmm'] = bgmm.fit_predict(embedding_2d)
#
# # 2. Visualisation du résultat
# plt.figure(figsize=(12, 8))
# scatter = plt.scatter(
#     embedding_2d[:, 0],
#     embedding_2d[:, 1],
#     c=df_umap['cluster_gmm'],
#     cmap='tab10',
#     s=10,
#     alpha=0.7
# )
# plt.colorbar(scatter, label='Cluster ID')
# plt.title("Clustering GMM (Basé sur la structure de Triage)")
# plt.show()
#
# # 3. L'AUDIT : Qu'est-ce qui change entre les clusters ?
# # On regarde la moyenne de chaque variable par cluster
# audit = df_umap.groupby('cluster_gmm').mean()
#
# print("📊 PROFILAGE DES CLUSTERS (Valeurs moyennes) :")
# # On transpose pour que ce soit plus lisible (Lignes = Variables, Colonnes = Clusters)
# display(audit.T.style.background_gradient(axis=1, cmap='YlOrRd'))

In [ ]:
# # DBSCAN
#
# from sklearn.cluster import DBSCAN
#
# # On essaye d'abord un eps qui capture les filaments
# # Si ton graph K-NN (le précédent) montrait un coude vers 0.3 ou 0.4 :
# db = DBSCAN(eps=0.4, min_samples=15)
# clusters = db.fit_predict(embedding_2d)
#
# # Visualisation
# plt.figure(figsize=(12, 8))
# plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=clusters, cmap='Spectral', s=5)
# plt.title(f"Clusters trouvés par DBSCAN : {len(np.unique(clusters)) - 1}")
# plt.show()

In [ ]:
# from scipy.cluster.hierarchy import dendrogram, linkage
#
# # On utilise la méthode 'ward' ou 'complete' sur tes coordonnées UMAP
# Z = linkage(embedding_2d, method='ward')
#
# plt.figure(figsize=(12, 6))
# dendrogram(Z, truncate_mode='lastp', p=20)  # On affiche les 20 derniers regroupements
# plt.title("Dendrogramme : Aide au choix du nombre de clusters")
# plt.ylabel("Distance de fusion")
# plt.axhline(y=15, color='r', linestyle='--')  # Exemple de ligne de coupe   METTRE A 520 POUR 10 CLUSTERS
# plt.show()

In [ ]:
# # ===========================================================
# # AVEC VARAIBLE TRIAGE SEULEMENT (SAUF TRI EVIDEMMENT) BINAIRE + VALEURS NUMERIQUES
# # ================================================================
#
# base_cols = [
#     'age', 'sexe_binr', 'is_ta_measured', 'is_fc_measured', 'is_temp_measured',
#     'is_sat_measured', 'is_fr_measured', 'has_o2_support', 'is_gcs_measured',
#     'is_hgt_mmol_L_measured', 'is_pupil_right_measured',
#     'is_urine_dipstick_clean_measured', 'is_eval_douleur_measured',
#     'is_ethylotest_measured', 'is_hemocue_measured', 'tas', 'tad', 'tam', 'fc', 'temp', 'sat', 'fr', 'o2_flow',
#     'gcs', 'hgt_mmol_L', 'pupil_right', 'pupil_left', 'ethylotest',
#     'hemocue', 'eval_douleur', 'duree_triage_ioa_min',
# ]
#
# # Ajouter les dummies de transport
# transport_cols = [c for c in df_scaled.columns if
#                   c.startswith("transport_grouped_")]  # VOIR POUR MOTIF A RAJOUTER UNE FOIS TROUVER UNE SOLUTION
#
# urine_cols = [c for c in df_scaled.columns if
#               c.startswith("urine_dipstick_clean")]
#
# umap_cols = base_cols + transport_cols + urine_cols
#
# df_umap = df_scaled[umap_cols]
#
# # 1. Création du réducteur UMAP 2D
# # n_neighbors : balance entre structure locale et globale
# # min_dist : contrôle le tassement des points (0.1 est standard)
# reducer_2d = umap.UMAP(
#     n_neighbors=15,
#     min_dist=0.1,
#     n_components=2,
#     random_state=42
# )
#
# # 2. Calcul des coordonnées
# embedding_2d = reducer_2d.fit_transform(df_umap)
#
# # 3. Affichage
# plt.figure(figsize=(12, 8))
# plt.scatter(
#     embedding_2d[:, 0],
#     embedding_2d[:, 1],
#     s=15,  # Taille des points
#     alpha=0.6,  # Transparence pour voir les densités
#     c='indigo',  # Couleur unique pour l'instant
#     edgecolors='none'
# )
#
# plt.title("Carte UMAP des Urgences (Structure Naturelle des Patients)", fontsize=14)
# plt.xlabel("Dimension UMAP 1")
# plt.ylabel("Dimension UMAP 2")
# plt.grid(True, linestyle='--', alpha=0.3)
# plt.show()

In [ ]:
# from sklearn.mixture import GaussianMixture
# import matplotlib.pyplot as plt
#
# n_components = range(1, 15)
# bics = []
#
# for n in n_components:
#     gmm = GaussianMixture(n_components=n, random_state=42)
#     gmm.fit(embedding_2d)  # On teste sur tes coordonnées UMAP
#     bics.append(gmm.bic(embedding_2d))
#
# plt.plot(n_components, bics, 'bo-')
# plt.xlabel('Nombre de clusters')
# plt.ylabel('BIC Score')  # Le point le plus bas est le meilleur
# plt.title('Recherche du nombre optimal de clusters (BIC)')
# plt.show()

In [ ]:
# # GMM
#
#
# from sklearn.mixture import BayesianGaussianMixture
#
# # 1. Ajustement du modèle sur tes coordonnées UMAP
# # On en demande 10, le modèle "Bayesian" filtrera ceux qui ne sont pas assez denses
# bgmm = BayesianGaussianMixture(
#     n_components=10,
#     weight_concentration_prior=1e-3,
#     random_state=42
# )
# df_umap['cluster_gmm'] = bgmm.fit_predict(embedding_2d)
#
# # 2. Visualisation du résultat
# plt.figure(figsize=(12, 8))
# scatter = plt.scatter(
#     embedding_2d[:, 0],
#     embedding_2d[:, 1],
#     c=df_umap['cluster_gmm'],
#     cmap='tab10',
#     s=10,
#     alpha=0.7
# )
# plt.colorbar(scatter, label='Cluster ID')
# plt.title("Clustering GMM (Basé sur la structure de Triage)")
# plt.show()
#
# # 3. L'AUDIT : Qu'est-ce qui change entre les clusters ?
# # On regarde la moyenne de chaque variable par cluster
# audit = df_umap.groupby('cluster_gmm').mean()
#
# print("📊 PROFILAGE DES CLUSTERS (Valeurs moyennes) :")
# # On transpose pour que ce soit plus lisible (Lignes = Variables, Colonnes = Clusters)
# display(audit.T.style.background_gradient(axis=1, cmap='YlOrRd'))

In [ ]:
# # DBSCAN
#
# from sklearn.cluster import DBSCAN
#
# # On essaye d'abord un eps qui capture les filaments
# # Si ton graph K-NN (le précédent) montrait un coude vers 0.3 ou 0.4 :
# db = DBSCAN(eps=0.4, min_samples=15)
# clusters = db.fit_predict(embedding_2d)
#
# # Visualisation
# plt.figure(figsize=(12, 8))
# plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=clusters, cmap='Spectral', s=5)
# plt.title(f"Clusters trouvés par DBSCAN : {len(np.unique(clusters)) - 1}")
# plt.show()

In [ ]:
# from sklearn.cluster import DBSCAN
#
# # 1. Lancer DBSCAN sur ton embedding
# # Ajuste eps selon ton graphique de coude (ex: 0.3 ou 0.5)
# db = DBSCAN(eps=0.4, min_samples=15)
# df_umap['cluster_dbscan'] = db.fit_predict(embedding_2d)
#
# # 2. Nettoyage des doublons (pour éviter le KeyError précédent)
# df_audit_db = df_umap.loc[:, ~df_umap.columns.duplicated()].copy()
#
# # 3. Calcul de l'audit
# audit_db = df_audit_db.groupby('cluster_dbscan').mean()
#
# # 4. Affichage
# print(f"📊 Audit DBSCAN : {len(audit_db)} groupes détectés (le -1 est le bruit)")
# display(audit_db.T.style.background_gradient(axis=1, cmap='coolwarm'))

In [ ]:
# from scipy.cluster.hierarchy import dendrogram, linkage
#
# # On utilise la méthode 'ward' ou 'complete' sur tes coordonnées UMAP
# Z = linkage(embedding_2d, method='ward')
#
# plt.figure(figsize=(12, 6))
# dendrogram(Z, truncate_mode='lastp', p=20)  # On affiche les 20 derniers regroupements
# plt.title("Dendrogramme : Aide au choix du nombre de clusters")
# plt.ylabel("Distance de fusion")
# plt.axhline(y=15, color='r', linestyle='--')  # Exemple de ligne de coupe
# plt.show()

In [ ]:

# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import umap.umap_ as umap
# from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
# from sklearn.cluster import DBSCAN
# from scipy.cluster.hierarchy import dendrogram, linkage
# from joblib import Parallel, delayed
#
# # ================================================================
# # HELPER BIC — parallélisé et rapide (réutilisé partout)
# # ================================================================
# def compute_bic(embedding, n_range=range(1, 15), sample_size=5000, title='BIC'):
#     """Calcule et affiche le BIC sur un sous-échantillon, en parallèle."""
#     idx = np.random.choice(len(embedding), size=min(sample_size, len(embedding)), replace=False)
#     sample = embedding[idx]
#
#     def fit_one(n):
#         gmm = GaussianMixture(
#             n_components=n,
#             covariance_type='diag',
#             max_iter=100,
#             n_init=1,
#             #random_state=42
#         )
#         gmm.fit(sample)
#         print(f"  n={n} ✓")
#         return gmm.bic(sample)
#
#     print(f"🔄 BIC en cours ({len(n_range)} modèles, {len(sample)} points)...")
#     bics = Parallel(n_jobs=os.cpu_count() - 1)(
#         delayed(fit_one)(n) for n in n_range
#     )
#
#     plt.figure(figsize=(8, 4))
#     plt.plot(list(n_range), bics, 'bo-')
#     plt.xlabel('Nombre de clusters')
#     plt.ylabel('BIC Score')
#     plt.title(title)
#     plt.grid(True, linestyle='--', alpha=0.3)
#     plt.show()
#     return bics
#
#
# # ================================================================
# # HELPER UMAP — paramètres optimisés communs
# # ================================================================
# def make_umap(data, title="Carte UMAP"):
#     reducer = umap.UMAP(
#         n_neighbors=15,
#         min_dist=0.1,
#         n_components=2,
#         #random_state=42,
#         init='random',           # évite le warning spectral
#         n_jobs=os.cpu_count()-1, # utilise tous les cœurs sauf 1
#         low_memory=False,
#     )
#     emb = reducer.fit_transform(data)
#
#     plt.figure(figsize=(12, 8))
#     plt.scatter(emb[:, 0], emb[:, 1], s=15, alpha=0.6, c='indigo', edgecolors='none')
#     plt.title(title, fontsize=14)
#     plt.xlabel("Dimension UMAP 1")
#     plt.ylabel("Dimension UMAP 2")
#     plt.grid(True, linestyle='--', alpha=0.3)
#     plt.show()
#     return emb
#
#
# # ================================================================
# # HELPER BGMM — clustering + visualisation + audit
# # ================================================================
# def run_bgmm(embedding, df_target, col_name='cluster_gmm', n_components=10, title='GMM'):
#     bgmm = BayesianGaussianMixture(
#         n_components=n_components,
#         weight_concentration_prior=1e-3,
#         #random_state=42
#     )
#     labels = bgmm.fit_predict(embedding)
#     df_target = df_target.copy()
#     df_target[col_name] = labels
#
#     plt.figure(figsize=(12, 8))
#     scatter = plt.scatter(embedding[:, 0], embedding[:, 1],
#                           c=labels, cmap='tab10', s=10, alpha=0.7)
#     plt.colorbar(scatter, label='Cluster ID')
#     plt.title(title)
#     plt.show()
#
#     audit = df_target.groupby(col_name).mean(numeric_only=True)
#     print(f"📊 Profilage BGMM — {len(audit)} clusters :")
#     display(audit.T.style.background_gradient(axis=1, cmap='YlOrRd'))
#     return df_target, labels
#
#
# # ================================================================
# # HELPER DBSCAN — clustering + visualisation + audit
# # ================================================================
# def run_dbscan(embedding, df_target, eps=0.4, min_samples=15, col_name='cluster_dbscan', title='DBSCAN'):
#     db = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=os.cpu_count()-1)
#     clusters = db.fit_predict(embedding)
#     df_target = df_target.copy()
#     df_target[col_name] = clusters
#
#     plt.figure(figsize=(12, 8))
#     plt.scatter(embedding[:, 0], embedding[:, 1], c=clusters, cmap='Spectral', s=5)
#     plt.title(f"{title} — {len(np.unique(clusters)) - 1} clusters (−1 = bruit)")
#     plt.show()
#
#     df_audit = df_target.loc[:, ~df_target.columns.duplicated()].copy()
#     audit = df_audit.groupby(col_name).mean(numeric_only=True)
#     print(f"📊 Audit DBSCAN : {len(audit)} groupes (le -1 est le bruit)")
#     display(audit.T.style.background_gradient(axis=1, cmap='coolwarm'))
#     return df_target, clusters
#
#
# # ================================================================
# # HELPER DENDROGRAMME
# # ================================================================
# def run_dendrogram(embedding, cut_y=15, title='Dendrogramme'):
#     Z = linkage(embedding, method='ward')
#     plt.figure(figsize=(12, 6))
#     dendrogram(Z, truncate_mode='lastp', p=20)
#     plt.title(title)
#     plt.ylabel("Distance de fusion")
#     plt.axhline(y=cut_y, color='r', linestyle='--', label=f'Coupe à y={cut_y}')
#     plt.legend()
#     plt.grid(axis='y', linestyle='--', alpha=0.4)
#     plt.show()


In [ ]:

# # ================================================================
# # SECTION 1 — TOUTES LES VARIABLES
# # ================================================================
# print("=" * 60)
# print("SECTION 1 — Toutes les variables")
# print("=" * 60)
#
# embedding_2d = make_umap(X_scaled, title="UMAP — Toutes les variables")
# bics_1 = compute_bic(embedding_2d, title='BIC — Toutes les variables')
#
# #Décommentez une fois le n_optimal identifié sur le graphique BIC :
# n_optimal_1 = 6
# df_scaled, _ = run_bgmm(embedding_2d, df_scaled, col_name='cluster_gmm_all', n_components=10, title='BGMM — Toutes variables')
# df_scaled, _ = run_dbscan(embedding_2d, df_scaled, col_name='cluster_dbscan_all', title='DBSCAN — Toutes variables')
# run_dendrogram(embedding_2d, cut_y=15, title='Dendrogramme — Toutes variables')


In [ ]:
# # ================================================================
# # SECTION 2 — VARIABLES TRIAGE BINAIRES SEULEMENT
# # ================================================================
# print("=" * 60)
# print("SECTION 2 — Variables triage binaires")
# print("=" * 60)
#
# base_cols_bin = [
#     'age', 'sexe_binr', 'is_ta_measured', 'is_fc_measured', 'is_temp_measured',
#     'is_sat_measured', 'is_fr_measured', 'has_o2_support', 'is_gcs_measured',
#     'is_hgt_mmol_L_measured', 'is_pupil_right_measured',
#     'is_urine_dipstick_clean_measured', 'is_eval_douleur_measured',
#     'is_ethylotest_measured', 'is_hemocue_measured'
# ]
# transport_cols = [c for c in df_scaled.columns if c.startswith("transport_grouped_")]
# umap_cols_bin  = base_cols_bin + transport_cols
# df_umap_bin    = df_scaled[umap_cols_bin]
#
# embedding_2d_bin = make_umap(df_umap_bin, title="UMAP — Variables triage binaires")
# bics_2 = compute_bic(embedding_2d_bin, title='BIC — Variables triage binaires')
#
# #Décommentez une fois le n_optimal identifié :
# n_optimal_2 = 6
# df_umap_bin, _ = run_bgmm(embedding_2d_bin, df_umap_bin, col_name='cluster_gmm_bin', n_components=10, title='BGMM — Binaires')
# df_umap_bin, _ = run_dbscan(embedding_2d_bin, df_umap_bin, col_name='cluster_dbscan_bin', title='DBSCAN — Binaires')
# run_dendrogram(embedding_2d_bin, cut_y=520, title='Dendrogramme — Binaires')  # cut_y=520 pour ~10 clusters


In [ ]:

# ================================================================
# SECTION 3 — VARIABLES TRIAGE BINAIRES + NUMÉRIQUES
# ================================================================
# print("=" * 60)
# print("SECTION 3 — Variables triage binaires + numériques")
# print("=" * 60)
#
# base_cols_full = base_cols_bin + [
#     'tas', 'tad', 'tam', 'fc', 'temp', 'sat', 'fr', 'o2_flow',
#     'gcs', 'hgt_mmol_L', 'pupil_right', 'pupil_left', 'ethylotest',
#     'hemocue', 'eval_douleur', 'duree_triage_ioa_min',
# ]
# urine_cols     = [c for c in df_scaled.columns if c.startswith("urine_dipstick_clean")]
# umap_cols_full = base_cols_full + transport_cols + urine_cols
# df_umap_full   = df_scaled[umap_cols_full]
#
# embedding_2d_full = make_umap(df_umap_full, title="UMAP — Binaires + Numériques")
# bics_3 = compute_bic(embedding_2d_full, title='BIC — Binaires + Numériques')
#
# # Décommentez une fois le n_optimal identifié :
# n_optimal_3 = 6
# df_umap_full, _ = run_bgmm(embedding_2d_full, df_umap_full, col_name='cluster_gmm_full', n_components=10, title='BGMM — Binaires + Num')
# df_umap_full, _ = run_dbscan(embedding_2d_full, df_umap_full, col_name='cluster_dbscan_full', title='DBSCAN — Binaires + Num')
# run_dendrogram(embedding_2d_full, cut_y=15, title='Dendrogramme — Binaires + Num')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap
import numba

# 1. Métriques Hybrides (Jaccard vs Hamming)
@numba.njit()
def custom_mixed_jaccard(a, b, n_idx, b_idx):
    intersection = 0.0
    union = 0.0
    for i in b_idx:
        if a[i] == 1.0 or b[i] == 1.0:
            union += 1.0
            if a[i] == 1.0 and b[i] == 1.0: intersection += 1.0
    dist_bin = 1.0 - (intersection / union) if union > 0 else 0.0
    dist_num = 0.0
    for i in n_idx: dist_num += abs(a[i] - b[i])
    if len(n_idx) > 0: dist_num = (dist_num / len(n_idx)) / 4.0
    return (0.7 * dist_bin) + (0.3 * dist_num)

@numba.njit()
def custom_mixed_hamming(a, b, n_idx, b_idx):
    diff_bin = 0.0
    for i in b_idx:
        if a[i] != b[i]: diff_bin += 1.0
    dist_bin = diff_bin / len(b_idx) if len(b_idx) > 0 else 0.0
    dist_num = 0.0
    for i in n_idx: dist_num += abs(a[i] - b[i])
    if len(n_idx) > 0: dist_num = (dist_num / len(n_idx)) / 4.0
    return (0.7 * dist_bin) + (0.3 * dist_num)

if 'embeddings' not in locals():
    embeddings = {}

In [ ]:
# 2 : Fonctions de Réduction et Diagnostic

def diagnostic_sparsity(df, cols_to_check):
    bin_cols = [c for c in cols_to_check if df[c].nunique() <= 2]
    if not bin_cols: return
    presence_rates = (df[bin_cols] == 1).mean().sort_values(ascending=False)
    sparsity = (df[bin_cols] == 0).mean().mean()

    plt.figure(figsize=(10, len(bin_cols) * 0.3 + 2))
    sns.barplot(x=presence_rates.values, y=presence_rates.index, palette='viridis')
    plt.axvline(x=0.5, color='red', linestyle='--')
    plt.title(f'Sparsité du groupe : {sparsity:.1%}')
    plt.show()


def run_pca(X):
    pca = PCA(n_components=2, random_state=42)
    emb = pca.fit_transform(X)
    return emb


def run_tsne(X):
    # Pré-réduction PCA pour le t-SNE
    X_pre = PCA(n_components=min(50, X.shape[1])).fit_transform(X)
    return TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42, n_jobs=128).fit_transform(X_pre)


def run_umap_hybrid(X_df, n_idx, b_idx, metric_func):
    reducer = umap.UMAP(
        n_neighbors=80,
        min_dist=0.4,
        metric=metric_func,
        metric_kwds={'n_idx': n_idx, 'b_idx': b_idx},
        init='pca',
        # random_state=42,
        n_jobs=128
    )
    return reducer.fit_transform(X_df.values)


def plot_embedding(emb, title, color_col=None):
    plt.figure(figsize=(10, 7))
    sc = plt.scatter(emb[:, 0], emb[:, 1], c=color_col, cmap='Spectral', s=8, alpha=0.6)
    if color_col is not None: plt.colorbar(sc)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()


import plotly.express as px


def run_umap_3d_hybrid_jaccard(df_scaled_sub, n_idx, b_idx, title="UMAP 3D"):
    # 1. Calcul UMAP
    reducer = umap.UMAP(
        n_components=3, n_neighbors=80, min_dist=0.4,
        metric=custom_mixed_jaccard,
        metric_kwds={'n_idx': n_idx, 'b_idx': b_idx},
        init='pca', #random_state=42,
        n_jobs=128
    )
    emb_3d_jaccard = reducer.fit_transform(df_scaled_sub.values)

    # 2. Préparation DataFrame
    plot_df = pd.DataFrame(emb_3d_jaccard, columns=['x', 'y', 'z'])
    plot_df['tri_class'] = df_imputed.loc[df_scaled_sub.index, 'tri'].astype(int)

    # 3. ON FORCE LES COULEURS (Dictionnaire de codes Hex)
    # Tri 1: Rouge vif, Tri 2: Orange, Tri 3: Jaune, Tri 4: Vert clair, Tri 5: Vert foncé
    med_colors = {
        1: "#D7191C",  # Rouge Urgence
        2: "#FDAE61",  # Orange
        3: "#FFFFBF",  # Jaune
        4: "#A6D96A",  # Vert clair
        5: "#1A9641"  # Vert foncé
    }

    # On crée une colonne 'color_hex' basée sur le dictionnaire
    plot_df['color_hex'] = plot_df['tri_num'].map(med_colors)
    plot_df['tri_label'] = plot_df['tri_num'].astype(str)

    # On trie pour la légende
    plot_df = plot_df.sort_values('tri_num')

    # 4. Création du graphique
    fig_3d_jaccard = px.scatter_3d(
        plot_df, x='x', y='y', z='z',
        color='tri_label',
        color_discrete_map={str(k): v for k, v in med_colors.items()},
        category_orders={"tri_label": ["1", "2", "3", "4", "5"]},
        title=title,
        opacity=0.7
    )

    fig_3d_jaccard.update_traces(marker=dict(size=2))
    fig_3d_jaccard.show()

    return emb_3d_jaccard, fig_3d_jaccard



def run_umap_3d_hybrid_jaccard(df_scaled_sub, n_idx, b_idx, title="UMAP 3D"):
    # 1. Calcul UMAP
    reducer = umap.UMAP(
        n_components=3, n_neighbors=80, min_dist=0.4,
        metric=custom_mixed_jaccard,
        metric_kwds={'n_idx': n_idx, 'b_idx': b_idx},
        init='pca', #random_state=42,
        n_jobs=128
    )
    emb_3d_jaccard = reducer.fit_transform(df_scaled_sub.values)
    # 2. Préparation DataFrame
    plot_df = pd.DataFrame(emb_3d_jaccard, columns=['x', 'y', 'z'])
    plot_df['tri_num'] = df_imputed.loc[df_scaled_sub.index, 'tri'].astype(int)
    # 3. ON FORCE LES COULEURS (Dictionnaire de codes Hex)
    # Tri 1: Rouge vif, Tri 2: Orange, Tri 3: Jaune, Tri 4: Vert clair, Tri 5: Vert foncé
    med_colors = {
        1: "#D7191C",  # Rouge Urgence
        2: "#FDAE61",  # Orange
        3: "#FFFFBF",  # Jaune
        4: "#A6D96A",  # Vert clair
        5: "#1A9641"  # Vert foncé
    }
    # On crée une colonne 'color_hex' basée sur le dictionnaire
    plot_df['color_hex'] = plot_df['tri_num'].map(med_colors)
    plot_df['tri_label'] = plot_df['tri_num'].astype(str)
    # On trie pour la légende
    plot_df = plot_df.sort_values('tri_num')
    # 4. Création du graphique
    fig_3d_jaccard = px.scatter_3d(
        plot_df, x='x', y='y', z='z',
        color='tri_label',
        color_discrete_map={str(k): v for k, v in med_colors.items()},
        category_orders={"tri_label": ["1", "2", "3", "4", "5"]},
        title=title,
        opacity=0.7
    )
    fig_3d_jaccard.update_traces(marker=dict(size=2))
    fig_3d_jaccard.show()
    return emb_3d_jaccard, fig_3d_jaccard




def run_umap_3d_hybrid_hamming(df_scaled_sub, n_idx, b_idx, title="UMAP 3D"):
    # 1. Calcul UMAP
    reducer = umap.UMAP(
        n_components=3, n_neighbors=80, min_dist=0.4,
        metric=custom_mixed_hamming,
        metric_kwds={'n_idx': n_idx, 'b_idx': b_idx},
        init='pca', #random_state=42,
        n_jobs=128
    )
    emb_3d_hamming = reducer.fit_transform(df_scaled_sub.values)

    # 2. Préparation DataFrame
    plot_df = pd.DataFrame(emb_3d_hamming, columns=['x', 'y', 'z'])
    plot_df['tri_num'] = df_imputed.loc[df_scaled_sub.index, 'tri'].astype(int)

    # 3. ON FORCE LES COULEURS (Dictionnaire de codes Hex)
    # Tri 1: Rouge vif, Tri 2: Orange, Tri 3: Jaune, Tri 4: Vert clair, Tri 5: Vert foncé
    med_colors = {
        1: "#D7191C",  # Rouge Urgence
        2: "#FDAE61",  # Orange
        3: "#FFFFBF",  # Jaune
        4: "#A6D96A",  # Vert clair
        5: "#1A9641"  # Vert foncé
    }

    # On crée une colonne 'color_hex' basée sur le dictionnaire
    plot_df['color_hex'] = plot_df['tri_num'].map(med_colors)
    plot_df['tri_label'] = plot_df['tri_num'].astype(str)

    # On trie pour la légende
    plot_df = plot_df.sort_values('tri_num')

    # 4. Création du graphique
    fig_3d_hamming = px.scatter_3d(
        plot_df, x='x', y='y', z='z',
        color='tri_label',
        color_discrete_map={str(k): v for k, v in med_colors.items()},
        category_orders={"tri_label": ["1", "2", "3", "4", "5"]},
        title=title,
        opacity=0.7
    )

    fig_3d_hamming.update_traces(marker=dict(size=2))
    fig_3d_hamming.show()

    return emb_3d_hamming, fig_3d_hamming

In [ ]:
# 3 : Définition des Groupes


groups = {
    'Toutes variables': list(df_scaled.columns),

    'Triage binaires': [
        'age', 'sexe_binr', 'is_ta_measured', 'is_fc_measured', 'is_temp_measured',
        'is_sat_measured', 'is_fr_measured', 'has_o2_support', 'is_gcs_measured',
        'is_hgt_mmol_L_measured', 'is_pupil_right_measured',
        'is_urine_dipstick_clean_measured', 'is_eval_douleur_measured',
        'is_ethylotest_measured', 'is_hemocue_measured',
        *[c for c in df_scaled.columns if c.startswith('transport_grouped_')]
    ],

    'Triage binaires + numériques': [
        'age', 'sexe_binr', 'is_ta_measured', 'is_fc_measured', 'is_temp_measured',
        'is_sat_measured', 'is_fr_measured', 'has_o2_support', 'is_gcs_measured',
        'is_hgt_mmol_L_measured', 'is_pupil_right_measured',
        'is_urine_dipstick_clean_measured', 'is_eval_douleur_measured',
        'is_ethylotest_measured', 'is_hemocue_measured',
        'tas', 'tad', 'fc', 'temp', 'sat', 'fr', 'o2_flow',
        'gcs', 'hgt_mmol_L', 'pupil_right', 'pupil_left', 'ethylotest',
        'hemocue', 'eval_douleur', 'duree_triage_ioa_min',
        *[c for c in df_scaled.columns if c.startswith('transport_grouped_')],
        *[c for c in df_scaled.columns if c.startswith('urine_dipstick_clean')]
    ],

    'Conso_soins': [
        'has_echo', 'has_scanner', 'has_radio_conv', 'has_irm', 'has_radio_interv', 'has_med_nucl', 'imagerie',
        'is_bilan_routine', 'is_bilan_cardio_vasc', 'is_bilan_infectieux', 'is_bilan_hepato_dig', 'is_bilan_calcique',
        'is_bilan_martial', 'is_bilan_coag_specifique',
        'is_gds', 'is_lcr', 'is_lactates', 'is_labo', 'devenir'
    ]

}

In [ ]:
# sparcity

for name, cols in groups.items():
    valid_cols = [c for c in cols if c in df_scaled.columns]
    print(f"\n--- Diagnostic : {name} ---")
    diagnostic_sparsity(df_scaled, valid_cols)

In [ ]:
# pca

for name, cols in groups.items():
    valid_cols = [c for c in cols if c in df_scaled.columns]
    X = df_scaled[valid_cols].values
    print(f"Calcul PCA : {name}...")
    embeddings[f'{name}_pca'] = run_pca(X)
    plot_embedding(embeddings[f'{name}_pca'], f'PCA - {name}', color_col=df_imputed['tri'])

In [ ]:
# tsne samplé

for name, cols in groups.items():
    valid_cols = [c for c in cols if c in df_scaled.columns]
    X = df_scaled[valid_cols].values

    print(f"Calcul t-SNE : {name}...")
    idx = np.random.choice(len(X), min(10000, len(X)), replace=False)
    emb_tsne = run_tsne(X[idx])
    plot_embedding(emb_tsne, f't-SNE - {name} (10k pts)', color_col=df_imputed['tri'].iloc[idx])

In [ ]:
# umap hybride

for name, cols in groups.items():
    valid_cols = [c for c in cols if c in df_scaled.columns]
    X_df = df_scaled[valid_cols]

    # Identification des indices
    n_idx = np.array([i for i, c in enumerate(valid_cols) if c in quanti_cols], dtype=np.int64)
    b_idx = np.array([i for i in range(len(valid_cols)) if i not in n_idx], dtype=np.int64)

    print(f"\n{'='*40}\nUMAP HYBRIDE : {name}\n{'='*40}")

    # 1. Jaccard (Focus Présence)
    print(f"→ Jaccard...")
    key_j = f'{name}_umap_jaccard'
    embeddings[key_j] = run_umap_hybrid(X_df, n_idx, b_idx, custom_mixed_jaccard)
    plot_embedding(embeddings[key_j], f'UMAP Jaccard - {name}', color_col=df_imputed['tri'])

    # 2. Hamming (Focus Accord Global)
    print(f"→ Hamming...")
    key_h = f'{name}_umap_hamming'
    embeddings[key_h] = run_umap_hybrid(X_df, n_idx, b_idx, custom_mixed_hamming)
    plot_embedding(embeddings[key_h], f'UMAP Hamming - {name}', color_col=df_imputed['tri'])

In [ ]:
# =======================================
# Appel umap 3d
# =======================================
# ==============================================================================
# BOUCLE D'APPEL 3D POUR LES 4 GROUPES (MÉTRIQUE HAMMING)
# ==============================================================================

# On cible tes 4 groupes
target_groups_3d = ['Toutes variables', 'Triage binaires', 'Triage binaires + numériques', 'Conso_soins']

for g_name in target_groups_3d:
    print(f"\n{'*'*50}")
    print(f"LANCEMENT UMAP 3D HAMMING : {g_name}")
    print(f"{'*'*50}")

    # 1. Sélection des colonnes présentes dans ton DF
    cols_3d = [c for c in groups[g_name] if c in df_scaled.columns]
    X_df_3d = df_scaled[cols_3d]

    # 2. Identification des indices (Quanti vs Binaires)
    # Important : s'assure que n_idx et b_idx sont recalculés pour chaque groupe
    n_idx_3d = np.array([i for i, c in enumerate(cols_3d) if c in quanti_cols], dtype=np.int64)
    b_idx_3d = np.array([i for i in range(len(cols_3d)) if i not in n_idx_3d], dtype=np.int64)

    # 3. EXÉCUTION DE TA FONCTION
    emb_3d_res, fig_3d_res = run_umap_3d_hybrid_hamming(
        X_df_3d,
        n_idx_3d,
        b_idx_3d,
        title=f"Exploration 3D Hamming : {g_name} ({len(cols_3d)} vars)"
    )

    # 4. Stockage des coordonnées 3D
    embeddings[f'{g_name}_umap_3d_hamming'] = emb_3d_res

    # 5. Sauvegarde du HTML (Nom de fichier propre)
    clean_name = g_name.replace(' ', '_').replace('+', 'and')
    filename = f"3D_Hamming_{clean_name}.html"
    fig_3d_res.write_html(filename)

    print(f"✅ Groupe [{g_name}] terminé. Fichier sauvegardé : {filename}")

print("\n✨ Félicitations ! Tes 4 vues 3D interactives sont prêtes.")

## A. CLUSTERING SUR UMAP 3D SUR TOUTES LES VARIABLES (HAMMING) ##

### 1. BGMM

In [ ]:
 # BGMM SUR UMAP 3D (HAMMING)

from sklearn.mixture import BayesianGaussianMixture

# 1. On s'assure que X_emb est bien un array propre
X_emb = embeddings['Toutes variables_umap_3d_hamming']

print("Lancement du BGMM Turbo...")

# 2. On bride les itérations et on simplifie
bgmm = BayesianGaussianMixture(
    n_components=10,
    covariance_type='diag',      # 'diag' est BEAUCOUP plus rapide que 'full'
    max_iter=100,                # On limite le nombre de tours de boucle
    n_init=1,                    # On ne le fait qu'une seule fois
    weight_concentration_prior=1e-2,
    random_state=42,
    init_params='kmeans'         # Initialisation plus rapide
)

# 3. Fit & Predict
df_imputed['cluster_bgmm'] = bgmm.fit_predict(X_emb)

print(f"✅ Terminé ! Clusters : {len(np.unique(df_imputed['cluster_bgmm']))}")

In [ ]:
import plotly.express as px
import pandas as pd

# ==============================================================================
# VISUALISATION 3D DES CLUSTERS BGMM (Basée sur l'Embedding)
# ==============================================================================

# 1. Création du DataFrame de plotting
# On récupère les coordonnées X, Y, Z de ton embedding "Toutes variables"
plot_df_bgmm = pd.DataFrame(X_emb, columns=['UMAP 1', 'UMAP 2', 'UMAP 3'])

# 2. Ajout des métadonnées pour le survol (hover) et la couleur
plot_df_bgmm['Cluster'] = df_imputed['cluster_bgmm'].astype(str)
plot_df_bgmm['Tri_Réel'] = df_imputed['tri'].astype(str)
plot_df_bgmm['Age'] = df_imputed['age']

# On trie par cluster pour que la légende soit ordonnée
plot_df_bgmm = plot_df_bgmm.sort_values('Cluster')

# 3. Création de la figure 3D
fig_3d_clusters = px.scatter_3d(
    plot_df_bgmm,
    x='UMAP 1', y='UMAP 2', z='UMAP 3',
    color='Cluster',
    symbol='Cluster', # Optionnel : change la forme des points par cluster
    title="Segmentation des Patients par BGMM (sur Embedding UMAP Hamming)",
    hover_data=['Tri_Réel', 'Age'], # Ce que tu vois en passant la souris
    color_discrete_sequence=px.colors.qualitative.Alphabet, # Palette très contrastée
    opacity=0.8
)

# 4. Ajustement cosmétique
fig_3d_clusters.update_traces(marker=dict(size=2.5)) # Points fins pour voir la structure
fig_3d_clusters.update_layout(
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(title="Groupes identifiés", itemsizing='constant')
)

# 5. Affichage et Sauvegarde
fig_3d_clusters.show()
fig_3d_clusters.write_html("resultat_final_clusters_bgmm_3d.html")

print("🚀 La figure 3D des clusters est prête ! Ouvre le fichier HTML pour explorer.")

### 2. HDBSCAN

In [ ]:
# HDBSCAN SUR UMAP 3D (HAMMING)


import hdbscan

# 1. Initialisation (min_cluster_size = taille mini d'un groupe de patients)
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=250,
    min_samples=20, # a tester avec differentes valeurs pour voir l'impact sur le nombre de clusters et de bruit
    prediction_data=True
)

# 2. Fit sur ton embedding UMAP 3D
df_imputed['cluster_hdbscan'] = clusterer.fit_predict(X_emb)

# 3. Combien de clusters et combien de "bruit" ?
n_clusters = len(set(clusterer.labels_)) - (1 if -1 in clusterer.labels_ else 0)
n_noise = list(clusterer.labels_).count(-1)

print(f"HDBSCAN a trouvé {n_clusters} clusters.")
print(f"Il a identifié {n_noise} patients comme 'bruit' (Outliers).")

In [ ]:
import plotly.express as px
import pandas as pd

# ==============================================================================
# VISUALISATION 3D : HDBSCAN (VUE MACRO)
# ==============================================================================

# 1. Création du DataFrame pour le plot
plot_df = pd.DataFrame(X_emb, columns=['UMAP 1', 'UMAP 2', 'UMAP 3'])
plot_df['Cluster'] = df_imputed['cluster_hdbscan'].astype(str)
plot_df['Tri_Réel'] = df_imputed['tri'].astype(str)

# 2. Gestion des couleurs : On veut que le bruit (-1) soit discret
# On utilise une palette qualitative pour les clusters
unique_clusters = sorted(plot_df['Cluster'].unique())
palette = px.colors.qualitative.Prism

# On crée la map de couleurs
color_map = {}
for i, cluster in enumerate(unique_clusters):
    if cluster == "-1":
        # ICI : On met les outliers en Rose Fluo (Magenta) pour qu'ils flashent
        color_map[cluster] = "#FFFFFF"
    else:
        color_map[cluster] = palette[i % len(palette)]

# 3. Génération de la figure 3D
fig_hdbscan = px.scatter_3d(
    plot_df,
    x='UMAP 1', y='UMAP 2', z='UMAP 3',
    color='Cluster',
    color_discrete_map=color_map,
    title=f"Segmentation HDBSCAN (min_size=350) : {n_clusters} Groupes | {n_noise} Outliers",
    hover_data=['Tri_Réel'],
    opacity=0.8
)

# 4. On réduit la taille des points pour voir à travers les nuages
fig_hdbscan.update_traces(marker=dict(size=2))

# 5. Affichage et Sauvegarde
fig_hdbscan.show()
fig_hdbscan.write_html("segmentation_hdbscan_macro_3d.html")

print("✅ Figure générée ! Ouvre 'segmentation_hdbscan_macro_3d.html' pour l'analyse.")

## B. CLUSTERING SUR LES VARIABLES DE TRIAGES ##

### BGMM

In [ ]:
# ==============================================================================
# CLUSTERING SUR LE GROUPE : Triage numérique + binaire
# ==============================================================================

# 1. On change la source de l'embedding
# Vérifie bien l'orthographe exacte dans ton dictionnaire 'embeddings'
key_name = 'Triage binaires + numériques_umap_3d_hamming'
X_emb_tri = embeddings[key_name]

print(f"🚀 Analyse lancée sur : {key_name}")

# --- A. BGMM (Les 10 clusters) ---
bgmm_triage = BayesianGaussianMixture(
    n_components=10,
    covariance_type='diag',
    max_iter=100,
    n_init=1,
    weight_concentration_prior=1e-2,
    random_state=42
)
df_imputed['cluster_bgmm_triage'] = bgmm_triage.fit_predict(X_emb_tri)

# ==============================================================================
# VISUALISATION 3D : BGMM (Sur Triage Numérique + Binaire)
# ==============================================================================

# 1. Préparation du DataFrame dédié au BGMM
plot_df_bgmm_triage = pd.DataFrame(X_emb_tri, columns=['UMAP 1', 'UMAP 2', 'UMAP 3'])
plot_df_bgmm_triage['Cluster_BGMM'] = df_imputed['cluster_bgmm_triage'].astype(str)
plot_df_bgmm_triage['Tri_Réel'] = df_imputed['tri'].astype(str)

# 2. Création de la figure BGMM
fig_bgmm_triage = px.scatter_3d(
    plot_df_bgmm_triage,
    x='UMAP 1', y='UMAP 2', z='UMAP 3',
    color='Cluster_BGMM',
    title=f"BGMM sur Triage Numérique + Binaire (12 Clusters forcés)",
    hover_data=['Tri_Réel'],
    template="plotly_dark", # On garde le fond noir pour la cohérence
    color_discrete_sequence=px.colors.qualitative.Alphabet,
    opacity=0.8
)

# 3. Ajustement de la taille des points
fig_bgmm_triage.update_traces(marker=dict(size=2))


# 4. Affichage et Sauvegarde
fig_bgmm_triage.show()
fig_bgmm_triage.write_html("bgmm_triage_numerique_binaire.html")

print("🚀 La vue BGMM est prête ! Tu peux maintenant comparer 'bgmm_triage...' et 'clusters_hdbscan...'")




### HDBSCAN

In [ ]:

# --- B. HDBSCAN (Les Outliers en Blanc) ---
clusterer_triage = hdbscan.HDBSCAN(
    min_cluster_size=900, # On augmente pour forcer des groupes plus gros
    min_samples=20,
    prediction_data=True
)
df_imputed['cluster_hdbscan_triage'] = clusterer_triage.fit_predict(X_emb_tri)

# Statistiques rapides
n_clusters_h = len(set(clusterer_triage.labels_)) - (1 if -1 in clusterer_triage.labels_ else 0)
n_noise_h = list(clusterer_triage.labels_).count(-1)
print(f"✅ HDBSCAN a trouvé {n_clusters_h} clusters et {n_noise_h} outliers blancs.")

# --- C. VISUALISATION 3D (Focus HDBSCAN + Outliers Blancs) ---
plot_df_triage = pd.DataFrame(X_emb_tri, columns=['UMAP 1', 'UMAP 2', 'UMAP 3'])
plot_df_triage['Cluster'] = df_imputed['cluster_hdbscan_triage'].astype(str)
plot_df_triage['Tri_Réel'] = df_imputed['tri'].astype(str)

unique_clusters = sorted(plot_df_triage['Cluster'].unique())
color_map = {c: px.colors.qualitative.Prism[i % 10] for i, c in enumerate(unique_clusters) if c != "-1"}
color_map["-1"] = "#FFFFFF" # On garde tes outliers en BLANC

fig_tri = px.scatter_3d(
    plot_df_triage, x='UMAP 1', y='UMAP 2', z='UMAP 3',
    color='Cluster',
    color_discrete_map=color_map,
    title=f"HDBSCAN sur Triage Numérique + Binaire ({n_noise_h} Outliers en Blanc)",
    hover_data=['Tri_Réel'],
    template="plotly_dark", # Fond noir pour que le blanc brille
    opacity=0.8
)

fig_tri.update_traces(marker=dict(size=2))
fig_tri.write_html("clusters_triage_numerique_binaire.html")
fig_tri.show()

In [ ]:
# On regarde les moyennes par cluster pour comprendre pourquoi il y en a 36
check_clusters = df_subset.groupby('cluster_hdbscan_triage').mean(numeric_only=True)

# On regarde l'écart-type entre les clusters :
# Si une variable a un écart-type énorme, c'est elle qui "tire" les clusters dans tous les sens.
print("Variables qui divisent le plus la population :")
print(check_clusters.std().sort_values(ascending=False).head(10))

In [ ]:
# # 2. Boucle de Comparaison des Métriques (Optimisée)
# # On va comparer les rendus sur un échantillon pour t'aider à choisir la meilleure stratégie visuelle.
#
# # Sélection du groupe de test
# target_group = 'Triage binaires + numériques'
# cols_test = [c for c in groups[target_group] if c in df_scaled.columns]
# X_test = df_scaled[cols_test].values
#
# # Identification dynamique des indices
# n_idx = np.array([i for i, c in enumerate(cols_test) if c in quanti_cols], dtype=np.int64)
# b_idx = np.array([i for i, c in enumerate(cols_test) if i not in n_idx], dtype=np.int64)
#
# # Échantillonnage pour la rapidité du test
# idx_sample = np.random.choice(len(X_test), 10000, replace=False)
# X_sample = X_test[idx_sample]
#
# metrics_to_test = {
#     'Euclidienne': {'metric': 'euclidean', 'kwds': {}},
#     'Jaccard Pur': {'metric': 'jaccard', 'kwds': {}}, # Attention: Jaccard pur n'aime pas les négatifs/continus
#     'Hybride (Jaccard+Num)': {'metric': custom_mixed_jaccard, 'kwds': {'n_idx': n_idx, 'b_idx': b_idx}}
# }
#
# fig, axes = plt.subplots(1, 3, figsize=(22, 7))
#
# for ax, (name, config) in zip(axes, metrics_to_test.items()):
#     print(f"Calcul UMAP : {name}...")
#     reducer = UMAP(
#         n_neighbors=50, min_dist=0.3,
#         metric=config['metric'],
#         metric_kwds=config['kwds'],
#         random_state=42, n_jobs=-1
#     )
#     emb = reducer.fit_transform(X_sample)
#
#     # Affichage coloré par TRI pour voir la pertinence clinique
#     sc = ax.scatter(emb[:, 0], emb[:, 1], c=df_imputed.loc[df_imputed.index[idx_sample], 'tri'],
#                     cmap='Spectral', s=5, alpha=0.6)
#     ax.set_title(f"UMAP: {name}")
#     plt.colorbar(sc, ax=ax)
#
# plt.tight_layout()
# plt.show()

Pourquoi choisir Jaccard plutôt que Hamming ?
Hamming compte les coïncidences (0-0 et 1-1).

Jaccard ne s'intéresse qu'aux présences (1-1).

L'intérêt pour toi : Si deux patients n'ont pas eu d'ECG, pas eu de radio et pas eu de transport par hélico (0-0-0), Hamming va dire qu'ils se ressemblent. Jaccard, lui, dira : "Je m'en fiche de ce qu'ils n'ont pas, je regarde s'ils partagent les mêmes symptômes ou examens réels".


. Pourquoi Jaccard est le favori des cliniciens
Dans ton cas, avoir un "1" (un examen fait, une mesure prise) est un événement actif décidé par un soignant.

L'asymétrie : Si deux patients ont eu une "Imagerie" et un "Labo", ils partagent un parcours de soins lourd.

Le problème du 0-0 : Avec Hamming, deux patients qui n'ont rien eu du tout (pas d'imagerie, pas de labo, pas de transport spécial) vont être jugés extrêmement similaires.

Le risque : Tu risques de créer un énorme "méga-cluster" de patients qui n'ont rien eu, ce qui va noyer les profils plus subtils. Jaccard évite cela en disant : "Le fait de ne pas avoir d'examen est l'état par défaut, ça ne suffit pas à dire que vous êtes identiques".

2. Pourquoi Hamming pourrait quand même te plaire
Tu as raison : "ne pas avoir eu tel examen" est une information.

Si un patient arrive avec un Tri 1 (Urgent) mais qu'il n'a pas d'imagerie, c'est une anomalie ou un profil très spécifique (ex: psychiatrie ou arrêt cardiaque immédiat).

Hamming capte cette "absence de soin" comme une caractéristique commune.

Le compromis idéal : La distance de Dice ou le Jaccard pondéré
Puisque tu hésites, il existe une règle d'or en clustering de triage : Utilise Jaccard si ton dataset contient beaucoup plus de "0" que de "1" (ce qui est souvent le cas en urgences : on ne fait pas d'imagerie à tout le monde).

Mon conseil pour ton projet :
Pour ton Bayesian GMM (étape 3), je te suggère de rester sur Jaccard. Pourquoi ?
Parce que ton but est d'identifier des "profils de patients". Un profil se définit mieux par ce qu'il est (vieux, tri 1, a eu un labo) que par ce qu'il n'est pas.

Le test ultime pour toi :
Regarde ton UMAP actuel (Hamming). Si tu vois un énorme bloc compact qui contient 60% de tes points, c'est que Hamming les a tous regroupés parce qu'ils "n'ont pas eu d'examens". Si tu veux "éclater" ce bloc pour voir les nuances, passe à Jaccard.

Comment tester sans tout changer ?
Modifie juste cette ligne dans ta fonction Numba et regarde si les clusters sur ton graphique UMAP deviennent plus "nets" ou s'ils se mélangent :


Gemini a dit
Wouah, alors là, le diagnostic est sans appel ! Ton graphique de présence montre une énorme majorité de zéros (sparsity globale de 17.2% de "1", donc plus de 80% de "0").

Toutes tes variables de constantes (is_ta_measured, is_fc_measured, etc.) ainsi que les examens biologiques sont quasiment à zéro dans ton échantillon.

Pourquoi l'UMAP Jaccard ne s'affiche pas ?
C'est mathématique : comme tu as énormément de lignes qui n'ont que des zéros sur les variables binaires, le calcul de Jaccard tombe sur 0/0 (pas d'intersection, pas d'union). Même avec la sécurité if union == 0, UMAP déteste avoir des milliers de points qui ont exactement la même distance nulle entre eux sur la partie binaire. Cela crée des instabilités numériques qui bloquent l'affichage ou font planter le moteur de rendu.

Ma recommandation : Le "Switch" Stratégique
Vu ton graphique, Jaccard est trop radical car il ignore 80% de ton tableau. Hamming est en fait plus adapté ici, car il va donner de l'importance au fait que le patient n'a pas eu de constantes mesurées (ce qui semble être le cas de la majorité).

Pour débloquer ton affichage et avoir de beaux clusters, fais ces trois modifications :

Reviens à Hamming (qui gérait mieux la masse de zéros).

Augmente min_dist : Tes points sont très "collés" car ils se ressemblent trop. Passe de 0.1 à 0.5.

Vérifie l'échelle de l'âge : Dans ton graphique Hamming précédent, on voyait des "traînées". C'est parce que l'âge n'était pas assez pondéré par rapport aux binaires.

Gemini a dit
Dans ta fonction mixed_metric_standardized, la distance utilisée pour l'âge est la Distance de Manhattan (aussi appelée distance absolue ou L
1
​
 ).

C'est cette partie du code :
dist_age = abs(a[age_idx] - b[age_idx]) / 2.0

Pourquoi c'est le meilleur choix pour ton cas ?
Linéarité : Contrairement à la distance Euclidienne (qui élève la différence au carré), la distance de Manhattan traite chaque année d'écart de la même manière. Un écart de 10 ans entre deux patients de 20 et 30 ans "pèsera" autant qu'un écart de 10 ans entre des patients de 70 et 80 ans.

Cohérence avec Hamming : La distance de Hamming (pour tes variables binaires) compte simplement le nombre de différences. Utiliser la distance absolue pour l'âge reste dans cette même philosophie de "somme directe" des écarts, ce qui rend ton mélange 0.6×Binaire+0.4×
A
ˆ
 ge très robuste.

Interprétation simple : Une distance de 0.5 sur l'âge signifie exactement que tes patients ont le maximum d'écart possible (l'un a -1, l'autre a 1 sur ton échelle standardisée).

Est-ce que c'est "Euclidien" ?
Techniquement, comme tu ne travailles que sur une seule dimension pour l'âge (une seule colonne), la distance de Manhattan (abs(a-b)) et la distance Euclidienne (
(a−b)
2


​
 ) donnent exactement le même résultat.

Le / 2.0 que nous avons ajouté est ce qui permet de transformer cette distance en une "proximité relative" compatible avec tes données binaires.

Résumé de ton calcul de distance finale :
Type de donnée	Variable	Formule de distance	Échelle finale
Binaire	Transport, Labo, etc.	Hamming (moyenne des différences)	[0,1]
Continue	Âge (standardisé)	Manhattan (Absolue) / 2	[0,1]
Pondération globale :

Distance=(0.6×Hamming)+(0.4×Manhattan_Age) jai fait 0.7 et 0.3



Gemini a dit
La différence visuelle majeure entre tes deux graphiques (le "violet" et le "bleu") s'explique par trois leviers mathématiques et techniques que tu as modifiés dans ton code.

Même si la logique globale reste la même, tu as changé la balance des poids, la sensibilité à l'âge et la stabilité du départ.

1. La pondération de l'Âge (Le changement d'échelle)
C'est le facteur le plus important pour la forme de tes clusters :

Code 1 (Violet) : Tu divisais la différence d'âge par 4.0. Comme ton âge est standardisé entre -1 et 1 (écart max de 2), la distance d'âge maximale était de 2/4=0.5. L'âge pesait donc très peu face au Hamming.

Code 2 (Bleu) : Tu divises par 2.0. La distance d'âge maximale est maintenant de 2/2=1.0. L'âge a désormais exactement le même poids potentiel que la partie binaire.

Résultat : Dans le deuxième code, l'âge "étire" beaucoup plus les groupes, créant ces traînées et ces filaments plus nets que tu vois dans le graphique bleu.

2. L'initialisation : random vs pca
C'est ce qui explique pourquoi la "carte" globale a changé de sens :

Code 1 (init='random') : UMAP place les points au hasard au début. Sur des données avec beaucoup de zéros (comme les tiennes), cela peut créer des amas un peu brouillons au centre.

Code 2 (init='pca') : UMAP commence par une structure stable basée sur les axes de variation les plus forts.

Résultat : Le graphique bleu est plus "organisé". On voit clairement un bloc principal et des archipels qui s'en détachent de manière plus logique. La PCA aide UMAP à comprendre que tes patients "sans examens" (les 80% de zéros) forment un socle commun.

3. La distance locale : min_dist et n_neighbors
Tu as modifié la "physique" de ton univers :

min_dist (0.3 → 0.5) : Tu as augmenté la distance minimale entre les points. C'est pour ça que le graphique bleu a l'air plus "aéré" et moins compact que le violet. Ça aide à éviter les gros tas de points illisibles.

n_neighbors (50 → 30) : En baissant ce chiffre, tu as demandé à UMAP de se concentrer davantage sur les détails très locaux. C'est ce qui fait apparaître les petits filaments très fins au lieu de grosses bulles.

Résumé des différences
Caractéristique	Code 1 (Violet)	Code 2 (Bleu)	Impact visuel
Poids de l'âge	Faible (divisé par 4)	Équilibré (divisé par 2)	Les points s'étirent selon l'âge
Départ (Init)	Aléatoire	PCA (Stable)	Structure globale plus cohérente
Espace (min_dist)	Serré (0.3)	Aéré (0.5)	Moins de chevauchement des points
Précision locale	Globale (50 voisins)	Détaillée (30 voisins)	Apparition de filaments fins
Lequel est le meilleur ? Le Code 2 (Bleu) est mathématiquement plus juste pour ton audit car il traite l'âge et le parcours de soins (binaires) sur une échelle comparable (0 à 1). C'est sur ce résultat que ton Bayesian GMM sera le plus efficace pour séparer, par exemple, les "Jeunes arrivés en perso" des "Âgés arrivés en ambulance".

Souhaites-tu que l'on passe à l'étape du GMM sur ce dernier résultat pour marquer officiellement les groupes ?